# D104 — Python Decorators

A decorator adds reusable behavior around a function without changing the function's core business logic.

## Learning goals

- Understand functions as first-class objects and decorators as higher-order functions.
- Build a decorator manually and then use `@decorator` syntax.
- Understand decoration time, call time, closures, and wrappers.
- Forward positional, keyword, default, and variable-length arguments.
- Preserve function metadata with `functools.wraps`.
- Implement cross-cutting concerns such as logging, timing, validation, authorization, retries, and caching.
- Build a timing decorator that logs start time, end time, and total duration.
- Build argument guards and parameterized decorator factories.

## 1. Functions are first-class objects

A Python function can be stored in a variable, passed to another function, and returned from a function. A **higher-order function** accepts a function, returns a function, or both.

In [ ]:
def greet(name):
    return f"Hello, {name}!"

another_name_for_greet = greet
print(another_name_for_greet("Asha"))

def apply(operation, value):
    return operation(value)

print(apply(greet, "Ravi"))

## 2. A decorator is a higher-order function

A function decorator normally:

1. receives the original function,
2. creates a wrapper function,
3. returns the wrapper.

The wrapper can run code before the original function, after it, or when an exception occurs.

In [ ]:
def announce(function):
    def wrapper():
        print("Before the function")
        result = function()
        print("After the function")
        return result
    return wrapper

def prepare_report():
    print("Preparing sales report")
    return {"status": "ready"}

decorated_report = announce(prepare_report)
report = decorated_report()
print(report)

The decorator is close to the higher-order function `apply`, but it has a different purpose:

- `apply(operation, value)` immediately uses a supplied function.
- `announce(function)` returns a new function with extra behavior.
- The closure inside `wrapper` remembers `function` even after `announce` has finished.

## 3. The `@decorator` syntax

The following definitions are equivalent:

```python
@announce
def prepare_report():
    ...

# Equivalent transformation:
prepare_report = announce(prepare_report)
```

Decoration happens when Python executes the `def` statement. The wrapper runs later, whenever the decorated function is called.

In [ ]:
def trace(function):
    print(f"Decorating {function.__name__}")

    def wrapper():
        print(f"Calling {function.__name__}")
        return function()

    return wrapper

@trace
def refresh_dashboard():
    print("Dashboard refreshed")

print("Function has been defined; now calling it")
refresh_dashboard()

## 4. Handling every kind of function argument

A reusable wrapper generally accepts `*args` and `**kwargs`:

- `args` collects positional arguments into a tuple.
- `kwargs` collects keyword arguments into a dictionary.
- `function(*args, **kwargs)` forwards both collections unchanged.

This handles positional, keyword, default, positional-only, keyword-only, `*args`, and `**kwargs` parameters of the wrapped function.

In [ ]:
from functools import wraps

def show_call(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        print(f"function={function.__name__}")
        print(f"positional arguments={args}")
        print(f"keyword arguments={kwargs}")
        result = function(*args, **kwargs)
        print(f"result={result!r}")
        return result
    return wrapper

@show_call
def create_order(customer, *items, priority="normal", discount=0, **metadata):
    subtotal = sum(items)
    return {
        "customer": customer,
        "items": items,
        "priority": priority,
        "total": subtotal * (1 - discount),
        "metadata": metadata,
    }

create_order(
    "C101",
    120,
    80,
    priority="high",
    discount=0.10,
    channel="web",
    region="south",
)

### Why must the wrapper return the result?

If the wrapper calls the original function but does not return its result, callers receive `None`. A transparent decorator should normally return exactly what the original function returns.

## 5. Preserve metadata with `functools.wraps`

Without `@wraps(function)`, the decorated function appears to be named `wrapper`; its docstring and annotations may also be hidden. `wraps` copies useful metadata and sets `__wrapped__`, which helps documentation, inspection, testing, and debugging tools.

In [ ]:
@show_call
def calculate_tax(amount: float, rate: float = 0.18) -> float:
    """Calculate tax for an amount."""
    return amount * rate

print(calculate_tax.__name__)
print(calculate_tax.__doc__)
print(calculate_tax.__annotations__)
print(calculate_tax.__wrapped__(1000, 0.05))

## 6. Cross-cutting concerns

A cross-cutting concern affects many otherwise unrelated functions. Common examples include:

- logging and audit trails,
- performance measurement,
- validation and authorization,
- retries and error reporting,
- caching and rate limiting.

Decorators keep this infrastructure behavior separate from business logic.

## 7. Logging calls and exceptions

In [ ]:
import logging
import sys

logger = logging.getLogger("decorator_lesson")
logger.handlers.clear()
handler = logging.StreamHandler(sys.stdout)
handler.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))
logger.addHandler(handler)
logger.setLevel(logging.INFO)
logger.propagate = False

def log_activity(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        logger.info("Starting %s", function.__name__)
        try:
            result = function(*args, **kwargs)
        except Exception:
            logger.exception("Failed %s", function.__name__)
            raise
        logger.info("Completed %s", function.__name__)
        return result
    return wrapper

@log_activity
def divide(total, count):
    return total / count

print(divide(100, 4))
try:
    divide(100, 0)
except ZeroDivisionError:
    print("Caller decided how to handle the error")

The decorator logs the failure and then uses bare `raise` to preserve the original exception and traceback. Logging should not silently convert a failure into `None`.

## 8. A `timeit`-like timing decorator

Use two clocks for two different jobs:

- `datetime.now()` gives human-readable start and end timestamps.
- `time.perf_counter()` is a high-resolution monotonic clock suitable for elapsed duration.

The `finally` block ensures that end time and duration are logged even if the wrapped function raises an exception.

In [ ]:
from datetime import datetime
from time import perf_counter, sleep

def log_timing(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        started_at = datetime.now().astimezone()
        start_counter = perf_counter()
        logger.info("%s | start=%s", function.__name__, started_at.isoformat(timespec="milliseconds"))

        try:
            return function(*args, **kwargs)
        finally:
            ended_at = datetime.now().astimezone()
            elapsed_seconds = perf_counter() - start_counter
            logger.info(
                "%s | end=%s | total_seconds=%.6f",
                function.__name__,
                ended_at.isoformat(timespec="milliseconds"),
                elapsed_seconds,
            )

    return wrapper

@log_timing
def load_partition(partition_name, delay=0.05):
    sleep(delay)
    return f"Loaded {partition_name}"

print(load_partition("2026-07-29", delay=0.02))

### Timing multiple calls and returning measurements

Logging is usually less disruptive than changing a function's return type. If a caller needs timing data programmatically, design that as an explicit API rather than unexpectedly returning `(result, duration)` from a general-purpose decorator.

## 9. Argument guarding with a decorator factory

Sometimes the decorator itself needs configuration. Calling `@require_positive("amount", "quantity")` first creates a decorator. This introduces three levels:

1. factory — receives configuration,
2. decorator — receives the function,
3. wrapper — receives call arguments.

`inspect.signature(...).bind(...)` maps positional and keyword values to their parameter names. `apply_defaults()` also fills in omitted default values.

In [ ]:
from inspect import signature

def require_positive(*parameter_names):
    def decorator(function):
        function_signature = signature(function)

        @wraps(function)
        def wrapper(*args, **kwargs):
            bound = function_signature.bind(*args, **kwargs)
            bound.apply_defaults()

            for name in parameter_names:
                if name not in function_signature.parameters:
                    raise TypeError(f"{function.__name__} has no parameter {name!r}")
                value = bound.arguments[name]
                if not isinstance(value, (int, float)) or isinstance(value, bool):
                    raise TypeError(f"{name} must be a number, received {type(value).__name__}")
                if value <= 0:
                    raise ValueError(f"{name} must be greater than zero, received {value}")

            return function(*args, **kwargs)

        return wrapper
    return decorator

@require_positive("unit_price", "quantity")
def invoice_line(product, unit_price, quantity=1):
    return {"product": product, "line_total": unit_price * quantity}

print(invoice_line("Keyboard", 2500, quantity=2))

for bad_call in [
    lambda: invoice_line("Keyboard", -2500, 2),
    lambda: invoice_line("Keyboard", "2500", 2),
]:
    try:
        bad_call()
    except (TypeError, ValueError) as error:
        print(type(error).__name__, "-", error)

## 10. A reusable predicate-based guard

A general guard can receive validation functions. This separates the validation mechanism from individual business rules.

In [ ]:
def guard_arguments(**rules):
    """Map parameter names to predicates that return True for valid values."""
    def decorator(function):
        function_signature = signature(function)

        unknown = set(rules) - set(function_signature.parameters)
        if unknown:
            raise TypeError(f"Unknown guarded parameters: {sorted(unknown)}")

        @wraps(function)
        def wrapper(*args, **kwargs):
            bound = function_signature.bind(*args, **kwargs)
            bound.apply_defaults()

            for name, predicate in rules.items():
                value = bound.arguments[name]
                if not predicate(value):
                    raise ValueError(f"Invalid value for {name}: {value!r}")

            return function(*args, **kwargs)

        return wrapper
    return decorator

@guard_arguments(
    email=lambda value: isinstance(value, str) and "@" in value,
    age=lambda value: isinstance(value, int) and not isinstance(value, bool) and 18 <= value <= 120,
)
def register_user(name, email, age):
    return f"Registered {name}"

print(register_user("Meera", "meera@example.com", 27))
try:
    register_user("Meera", "invalid-email", 15)
except ValueError as error:
    print(error)

Guards are useful at system boundaries, but they do not replace clear domain types or validation inside complex business objects. Avoid logging sensitive argument values such as passwords, tokens, or personal information.

## 11. Authorization as a cross-cutting concern

In [ ]:
def requires_role(required_role):
    def decorator(function):
        function_signature = signature(function)

        @wraps(function)
        def wrapper(*args, **kwargs):
            bound = function_signature.bind(*args, **kwargs)
            user = bound.arguments.get("user")
            if user is None or required_role not in user.get("roles", set()):
                raise PermissionError(f"Role {required_role!r} is required")
            return function(*args, **kwargs)

        return wrapper
    return decorator

@requires_role("admin")
def delete_staging_data(user, table_name):
    return f"Deleted staging data from {table_name}"

admin = {"name": "Dev", "roles": {"analyst", "admin"}}
analyst = {"name": "Isha", "roles": {"analyst"}}

print(delete_staging_data(admin, "raw_orders"))
try:
    delete_staging_data(analyst, "raw_orders")
except PermissionError as error:
    print(error)

This simplified example teaches the pattern. Production authorization should rely on trusted identity and policy data rather than a caller-controlled dictionary.

## 12. Retry decorator

Retries are appropriate only for transient failures and safe operations. Do not blindly retry every exception or repeat a non-idempotent operation that may already have succeeded.

In [ ]:
def retry(max_attempts=3, exceptions=(Exception,)):
    if max_attempts < 1:
        raise ValueError("max_attempts must be at least 1")

    def decorator(function):
        @wraps(function)
        def wrapper(*args, **kwargs):
            for attempt in range(1, max_attempts + 1):
                try:
                    return function(*args, **kwargs)
                except exceptions:
                    if attempt == max_attempts:
                        raise
                    logger.warning("%s failed; retrying (%d/%d)", function.__name__, attempt, max_attempts)
        return wrapper
    return decorator

attempt_state = {"count": 0}

@retry(max_attempts=3, exceptions=(ConnectionError,))
def fetch_exchange_rate():
    attempt_state["count"] += 1
    if attempt_state["count"] < 3:
        raise ConnectionError("Temporary service failure")
    return 83.75

print("Rate:", fetch_exchange_rate())

## 13. Caching with a standard decorator

Python already provides production-quality decorators. `functools.lru_cache` memoizes results based on arguments. Its arguments must be hashable, and it is best suited to deterministic functions.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=128)
def fibonacci(number):
    if number < 2:
        return number
    return fibonacci(number - 1) + fibonacci(number - 2)

print(fibonacci(30))
print(fibonacci.cache_info())

## 14. Stacking decorators

Decorators are applied from the bottom upward:

```python
@outer
@inner
def work():
    ...

# Equivalent to:
work = outer(inner(work))
```

At call time, the outer wrapper begins first, then the inner wrapper. Order matters: validation outside timing can reject a call before timing begins; timing outside validation measures validation as well.

In [ ]:
@log_activity
@log_timing
@require_positive("row_count")
def transform_rows(row_count):
    sleep(0.01)
    return row_count * 2

print("Transformed:", transform_rows(500))

## 15. Decorating methods

The same `*args, **kwargs` pattern works for methods. The instance (`self`) is simply the first positional argument.

In [ ]:
class OrderService:
    @log_timing
    def total(self, prices, tax_rate=0.18):
        subtotal = sum(prices)
        return subtotal * (1 + tax_rate)

service = OrderService()
print(service.total([100, 200, 300], tax_rate=0.05))

## 16. Common mistakes

- Returning `wrapper()` from the decorator calls it immediately; return `wrapper` instead.
- Forgetting `return function(...)` loses the original result.
- Omitting `*args, **kwargs` restricts the kinds of functions that can be decorated.
- Omitting `@wraps(function)` hides useful metadata.
- Catching and suppressing every exception changes the original function's contract.
- Mutating arguments or return values can make a decorator surprising.
- Logging all arguments can leak secrets or personal data.
- Excessive decorator stacking can make control flow difficult to follow.
- A normal wrapper around `async def` returns a coroutine; async decorators should usually define `async def wrapper(...)` and `await function(...)`.

## 17. Mental model

For this code:

```python
@log_timing
def load_data(source):
    return source.read()
```

Use this model:

```text
definition time: load_data = log_timing(original_load_data)

call time:
caller
  -> timing wrapper starts clock
      -> original_load_data runs
  <- timing wrapper stops clock and returns original result
<- caller receives result
```

## 18. Practice exercises

1. Write `@count_calls` so each decorated function exposes the number of times it has been called as `function.call_count`.
2. Write `@truncate_result(max_length=20)` for functions that return strings.
3. Extend `retry` with a configurable delay between attempts.
4. Create `@require_non_empty("customer_id")` using `inspect.signature`.
5. Stack a guard, logger, and timer. Predict the output order before running it.
6. Modify `log_timing` to send a dictionary of metrics to a supplied callback.

## 19. Summary

- A decorator is a higher-order function that transforms another callable.
- `@decorator` is syntax for rebinding a function name to the decorator's result.
- A closure lets the wrapper remember the original function and configuration.
- `*args` and `**kwargs` make wrappers broadly reusable.
- `functools.wraps` preserves the wrapped function's identity and metadata.
- Decorator factories add configuration such as roles, parameter names, retry counts, or thresholds.
- Decorators are well suited to cross-cutting concerns when they keep the wrapped function's contract clear.